# Notebook that does the processing of foraging session trajectories

### Import functions necessary to do the processing 

In [1]:
import os
import gc
import glob
import time
from processing_TowerCoordinates import *
from processing_session_trajectory import *

### Make a list of the mice and sessions to process
Define the folder where your MOUXXX folders are, generate a list of MOUXXX folders and for each mice a list of sessions

In [2]:
# indicate where the data are
# path_to_data_folder is the path of the folder where you store your different mice.

path_to_data_folder = '/media/david/Maud_T5/Thesis/HarddriveDatas/Batch_21_temporaryfolder'

pattern_of_MOU_Folders = os.path.join(path_to_data_folder, "MOU*")

# List all mice in the data folder (If you want to process all the mice in your data folder),
mice_list = [os.path.basename(path) for path in glob.glob(pattern_of_MOU_Folders)]
mice_list=sorted(mice_list)

# Print the number of mice, the list of mice, and add an empty line
print(f'Found {len(mice_list)} {"mice" if len(mice_list) > 1 else "mouse"} in the data folder: {", ".join(mice_list)}\n')


#If you want to process a subset of mice uncomment the line below and comment the 2 lines above

mice_list = ["MOU5470","MOU5471","MOU5472","MOU5480","MOU5481","MOU5482","MOU5464","MOU5465","MOU5466","MOU5467","MOU5468"]

session_list = {}
for mouse in mice_list:
    mouse_folder = os.path.join(path_to_data_folder,mouse)
    session_list[mouse] = sorted([name for name in os.listdir(mouse_folder)
                           if os.path.isdir(os.path.join(mouse_folder, name))
                           and name.startswith('MOU')])
    nb_sessions = len(session_list[mouse])
    print(f'Hello, I\'m {mouse}! I have foraged for {nb_sessions} sessions:')
    print(session_list[mouse], '\n')


Found 11 mice in the data folder: MOU5464, MOU5465, MOU5466, MOU5467, MOU5468, MOU5470, MOU5471, MOU5472, MOU5480, MOU5481, MOU5482

Hello, I'm MOU5470! I have foraged for 6 sessions:
['MOU5470_20260323-0858', 'MOU5470_20260323-1324', 'MOU5470_20260324-0932', 'MOU5470_20260324-1356', 'MOU5470_20260325-0904', 'MOU5470_20260325-1402'] 

Hello, I'm MOU5471! I have foraged for 6 sessions:
['MOU5471_20260323-0915', 'MOU5471_20260323-1341', 'MOU5471_20260324-0950', 'MOU5471_20260324-1416', 'MOU5471_20260325-0918', 'MOU5471_20260325-1419'] 

Hello, I'm MOU5472! I have foraged for 5 sessions:
['MOU5472_20260323-0933', 'MOU5472_20260323-1358', 'MOU5472_20260324-1007', 'MOU5472_20260324-1434', 'MOU5472_20260325-0932'] 

Hello, I'm MOU5480! I have foraged for 5 sessions:
['MOU5480_20260323-0951', 'MOU5480_20260323-1415', 'MOU5480_20260324-1025', 'MOU5480_20260324-1453', 'MOU5480_20260325-0946'] 

Hello, I'm MOU5481! I have foraged for 5 sessions:
['MOU5481_20260323-1011', 'MOU5481_20260323-1432',

## Process all the sessions in mice_list
### the key option here is wether to force processing or not (in case the data have already been processed)
#### for this the variable process should be set as true (to force) or false if the sessiobn has already been processed (if this is the case the name of the session has been saved in the  ListSessionsAnalyzed.txt file

In [3]:
# Add an overwrite flag
overwrite = False  # Set to True if you want to overwrite existing pickle files
bad_sessions = []

mice_to_process = mice_list

for mouse in mice_to_process:
    folder_path_mouse_to_process = os.path.join(path_to_data_folder, mouse)
    
    # Get the list of sessions
    sessions_to_process = sorted([name for name in os.listdir(folder_path_mouse_to_process)
                                  if os.path.isdir(os.path.join(folder_path_mouse_to_process, name))
                                  and name.startswith('MOU')])
    
    nb_sessions = len(sessions_to_process)
    print(f'Processing mouse {mouse}. There is/are {nb_sessions} sessions to process:')
    print(sessions_to_process, '\n')
    
    # Process each session
    for sessionindex,session_to_process in enumerate(sessions_to_process):
        print(f'Processing the trajectory of session {session_to_process}')
        if sessionindex==0: # for the first session of a given animal, we get the trapeze coordinates and then we will resuse them for the remaining sesions
            trapeze_width, towers_coordinates = get_trapeze_and_tower_data(folder_path_mouse_to_process, session_to_process)
            all_trapezes_coordinates_cm,towers_coordinates_cm= generate_trapeze_and_tower_coordinates(towers_coordinates, trapeze_width)
            
        
        # Define the pickle file path
        output_pickle_filename = f"{session_to_process}_basic_processing_output.pickle"
        output_pickle_filepath = os.path.join(folder_path_mouse_to_process, session_to_process, output_pickle_filename)
        
        # Check if the pickle file already exists
        if not overwrite and os.path.exists(output_pickle_filepath):
            print(f'Pickle file already exists for session {session_to_process}, skipping processing.')
            continue  # Skip processing if the file exists and overwrite is False
        
        # Run the processing if file doesn't exist or overwrite is True
        output=process_trajectory(folder_path_mouse_to_process, session_to_process,all_trapezes_coordinates_cm,towers_coordinates_cm)
        if output is not None:
            bad_sessions.append(output)
        print('#########################\n')

Processing mouse MOU5470. There is/are 6 sessions to process:
['MOU5470_20260323-0858', 'MOU5470_20260323-1324', 'MOU5470_20260324-0932', 'MOU5470_20260324-1356', 'MOU5470_20260325-0904', 'MOU5470_20260325-1402'] 

Processing the trajectory of session MOU5470_20260323-0858
Total time: 885.00 s.
The total distance is: 141.23 m
The average running speed is: 15.96 cm/s
Processing run_between_towers epochs...
Processing run_toward_tower epochs...
Processing exploratory_run epochs...
Processing immobility epochs...
Session processing results saved to /media/david/Maud_T5/Thesis/HarddriveDatas/Batch_21_temporaryfolder/MOU5470/MOU5470_20260323-0858/MOU5470_20260323-0858_basic_processing_output.pickle
#########################

Processing the trajectory of session MOU5470_20260323-1324
Total time: 884.96 s.
The total distance is: 156.15 m
The average running speed is: 17.65 cm/s
Processing run_between_towers epochs...
Processing run_toward_tower epochs...
Processing exploratory_run epochs...
P

## Verify that there is no bad sessions 

In [4]:
print(bad_sessions)

[]
